# NB 6 — Uncertainty and safe abstention
**Goal:** make an agent **know when it doesn't know**, and *abstain / escalate* instead of answering confidently.

We estimate confidence with **self-consistency**: ask the model the same thing several times (temperature > 0) and measure how much the answers agree. High agreement → answer; low agreement → hand off to a human. Every question is grounded in a short record, so a real model is reasoning over given facts — not inventing them. (Runs in MOCK mode with no API key; set a key to make it real.)

> ### In plain terms
> A trustworthy agent has to **know when it doesn't know.** The trick used here: ask the model the same question several times. If the answers **agree**, it's confident; if they **scatter**, hand the question to a human instead of guessing.
>
> Two important details: we compare answers by **meaning, not exact wording** (so "warfarin overdose" and "warfarin-related bleeding" count as the same idea), and every question is asked against one specific patient record.
>
> **What you'll see:** first a confident question vs. an uncertain one; then a table where you can watch the "confidence bar" (tau) move and turn answers into escalations. One honest caveat throughout: **agreement means the model is consistent, not that it's right.**

In [1]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses.
#   2) REAL model: pip install openai, then set env vars. OpenRouter example:
#        export OPENAI_BASE_URL=https://openrouter.ai/api/v1
#        export OPENAI_API_KEY=sk-or-...          # your OpenRouter key (never commit it)
#        export MODEL=openai/gpt-4o-mini          # any OpenRouter model id
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

import random
_RNG = random.Random(7)
# The model's latent answer spread per question keyword. In a REAL run you don't write
# these — you get the spread for free by sampling at temperature>0. Here they simulate it.
_DIST = {
    "creatinine":    {"1.2 mg/dL": 0.95, "1.9 mg/dL": 0.05},
    "potassium":     {"4.1 mmol/L": 0.85, "5.6 mmol/L": 0.15},
    "warfarin dose": {"5 mg": 0.75, "7.5 mg": 0.25},
    "chest pain":    {"stable angina": 0.45, "GERD": 0.3, "anxiety": 0.25},   # genuinely split
    "cause of the":  {"sepsis": 0.55, "PE": 0.25, "dehydration": 0.20},       # borderline
}
def _mock(messages, temperature=0):
    u = " ".join(m["content"] for m in messages if m["role"] == "user").lower()
    u = u.split("question:")[-1]          # match on the QUESTION, not the record above it
    for key, dist in _DIST.items():
        if key in u:
            return _RNG.choices(list(dist), weights=list(dist.values()))[0]
    return "unsure"

Backend: REAL model = openai/gpt-4o-mini


### A record, and confidence by agreement
Factual questions are answerable **from the record** (the model should be confident and right); diagnostic questions genuinely are not (it should be unsure). Confidence = the fraction of samples that agree with the majority answer.

In [2]:
from collections import Counter
import re

RECORD = ("58M, post-op day 5 after DVT. Labs today: creatinine 1.2 mg/dL, potassium 4.1 mmol/L, "
          "INR 4.2 (range 2.0-3.0). Medications: warfarin 5 mg daily, lisinopril 10 mg daily. "
          "Now reports intermittent chest pain with a nonspecific ECG and one episode of transient hypotension.")
SYS = ("You are a clinical assistant. Use ONLY the record provided. "
       "Answer with a short value or phrase only — no explanation.")

def ask_n(question, n=10):
    "Sample the model n times (temperature 1) on the same record + question."
    msgs = [{"role":"system","content":SYS},
            {"role":"user","content":f"Record:\n{RECORD}\n\nQuestion: {question}"}]
    return [chat(msgs, temperature=1) for _ in range(n)]

# --- Agreement by MEANING, not exact wording --------------------------------
# A real model phrases the same idea many ways ("warfarin overdose",
# "warfarin-related bleeding"). We map each answer to a canonical bucket so
# "agreement" measures whether the model MEANS the same thing, not whether it
# typed the same characters.
_BUCKETS = [
 ("cardiac / ischemia",         ["angina","cardiac","ischem","coronary","acs","myocard"]),
 ("anticoagulation / bleeding", ["warfarin","anticoag","bleed","coagulopath","inr","supratherap"]),
 ("GERD / reflux",              ["gerd","reflux","heartburn","esophag"]),
 ("anxiety / panic",            ["anxiety","panic","psychogenic"]),
 ("pulmonary embolism",         ["pulmonary embol","embolism"]),
 ("sepsis / infection",         ["sepsis","septic","infection"]),
 ("hypovolemia / dehydration",  ["dehydr","hypovol","volume depl"]),
 ("musculoskeletal",            ["musculoskelet","costochondr","muscle strain"]),
]
def canon(s):
    "Normalise a model answer to a comparable form (a value, or a clinical bucket)."
    s = s.strip().lower().rstrip(".")
    v = re.fullmatch(r"(?:the\s+)?(\d+(?:\.\d+)?)\s*(mg/dl|mmol/l|mg|mcg|units?|%)?", s)
    if v:                                    # a value answer, e.g. "1.2 mg/dl"
        return f"{float(v.group(1)):g} {(v.group(2) or '').replace(' ','')}".strip()
    for label, kws in _BUCKETS:              # a phrase answer -> clinical bucket
        if any(k in s for k in kws): return label
    return s
def matches(ans, correct):
    a, c = canon(ans), canon(correct)
    return c in a or a in c

def confidence(question, n=10):
    "Sample, group by meaning, return (top bucket, agreement fraction, raw samples)."
    raw = ask_n(question, n)
    buckets = [canon(s) for s in raw]
    top, cnt = Counter(buckets).most_common(1)[0]
    return top, cnt/len(buckets), raw

### One confident question, one genuinely uncertain one
Same machinery, two very different confidence signals. The creatinine is written in the record, so the model should agree with itself every time. The cause of the transient hypotension is *not* in the record, so honest sampling should scatter — and low agreement should route it to a human.

In [3]:
# Two questions: one answerable FROM THE RECORD, one that genuinely is not.
for q in ["What is the patient's most recent creatinine? (value only)",
          "What is the single most likely cause of the transient hypotension? (one phrase)"]:
    top, conf, raw = confidence(q, n=8)
    print("Q:", q)
    print("   raw answers        :", raw)
    print("   grouped by meaning :", dict(Counter(canon(s) for s in raw)))
    print(f"   majority idea      : {top!r}   agreement: {conf:.0%}")
    print(f"   action             : {'ANSWER' if conf>=0.7 else 'ESCALATE to a clinician'}\n")

Q: What is the patient's most recent creatinine? (value only)
   raw answers        : ['1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL', '1.2 mg/dL']
   grouped by meaning : {'1.2 mg/dl': 8}
   majority idea      : '1.2 mg/dl'   agreement: 100%
   action             : ANSWER



Q: What is the single most likely cause of the transient hypotension? (one phrase)
   raw answers        : ['Warfarin overdose', 'Warfarin overdose.', 'Warfarin overdose.', 'Warfarin-induced bleeding.', 'Warfarin overdose.', 'Warfarin overdose', 'Warfarin overdose.', 'Warfarin overdose']
   grouped by meaning : {'anticoagulation / bleeding': 8}
   majority idea      : 'anticoagulation / bleeding'   agreement: 100%
   action             : ANSWER



### The confidence bar is a dial — watch it move, per question
For each question we sample the model, group its answers by meaning, and get an **agreement %**. Then we sweep the bar (`tau`) from lax (0.5) to strict (0.9). Reading **down** a column shows the trade-off: raise the bar and more questions flip from **ANS** (answered) to **esc** (escalated to a clinician). Factual lookups sit near 100% and stay answered; open-ended questions sit lower and get escalated first — which is exactly what you want.

In [4]:
# One row per question: its agreement, then ANSWER/escalate at each threshold (tau).
QS = [
 ("creatinine  (factual)",    "What is the most recent creatinine? (value only)",  "1.2 mg/dL"),
 ("potassium   (factual)",    "What is the most recent potassium? (value only)",   "4.1 mmol/L"),
 ("warfarin    (factual)",    "What is the current warfarin dose? (value only)",   "5 mg"),
 ("chest-pain cause (open)",  "Single most likely cause of the chest pain? (one phrase)", None),
 ("hypotension cause (open)", "Single most likely cause of the transient hypotension? (one phrase)", None),
]
N, TAUS = 12, [0.5, 0.6, 0.7, 0.8, 0.9]

hdr = f"{'question':28}{'agree':>6}   " + "  ".join(f"tau {t:>3}" for t in TAUS)
print(hdr); print("-" * len(hdr))
for label, q, correct in QS:
    top, conf, _ = confidence(q, N)
    grid = "  ".join(f"{'ANS' if conf>=t else 'esc':>6}" for t in TAUS)
    tag = "" if correct is None else ("   [correct]" if matches(top, correct) else "   [WRONG]")
    print(f"{label:28}{conf:>5.0%}   {grid}{tag}")
print("\nANS = answered  ·  esc = escalated to a clinician.")
print("Read DOWN a tau column: raising the bar turns more answers into escalations.")
print("Factual lookups sit near 100% and stay ANSWERED; open questions sit lower and flip to esc first.")
print("Agreement = consistency, NOT truth — a high-agreement answer to an open question can be confidently wrong.")

question                     agree   tau 0.5  tau 0.6  tau 0.7  tau 0.8  tau 0.9
--------------------------------------------------------------------------------


creatinine  (factual)        100%      ANS     ANS     ANS     ANS     ANS   [correct]


potassium   (factual)        100%      ANS     ANS     ANS     ANS     ANS   [correct]


warfarin    (factual)        100%      ANS     ANS     ANS     ANS     ANS   [correct]


chest-pain cause (open)      100%      ANS     ANS     ANS     ANS     ANS


hypotension cause (open)     100%      ANS     ANS     ANS     ANS     ANS

ANS = answered  ·  esc = escalated to a clinician.
Read DOWN a tau column: raising the bar turns more answers into escalations.
Factual lookups sit near 100% and stay ANSWERED; open questions sit lower and flip to esc first.
Agreement = consistency, NOT truth — a high-agreement answer to an open question can be confidently wrong.


### Takeaway
Abstention turns a silent wrong answer into a routed one. As the threshold rises, escalations rise — that trade-off *is* the design decision, and it belongs to the clinical workflow, not the model. Grounding the questions in the record is what makes the confidence meaningful: the model is confident on what it was given and unsure on what it wasn't. Better uncertainty signals (calibrated probabilities, conformal prediction, semantic entropy) are exactly where a research program can add value.

*Try:* with a real model set, re-run — the spread now comes from the model itself; watch which questions it is quietly unsure about, and remember agreement is not the same as being right.